In [1]:
#Import library and print dataset and check thee size shape and info
import pandas as pd
import numpy as np

df = pd.read_csv(
    "AI_Usage_and_Impact_on_Students_and_Professionals.csv"
)

print(df.head())
print(df.shape)
print(df.info())

  User_ID Age  Gender         Country     User_Type Education_Level  \
0   U0269  30    Male  United Kingdom  Professional        Master's   
1   U0217  30    Male   United States  Professional      Bachelor's   
2   U0152  17    Male          Canada       Student             NaN   
3   U0264  35  FEMALE          Brazil  Professional      Bachelor's   
4   U0263  19  Female          Canada       Student   Undergraduate   

                                        Profession Monthly_Income  \
0                                 Business Analyst           5421   
1                      Customer Support Specialist           8903   
2                         Computer Science Student            402   
3  Desenvolvedora de Software / Software Developer           1367   
4                                  Biology Student            438   

         AI_Tool        AI_Purpose AI_Usage_Hours_Per_Day  \
0        ChatGPT       Office Work                  2 hrs   
1        ChatGPT  Customer Support  

# Now we check duplicate and null value

In [2]:
print(df.isnull().sum())
print("Total missing:", df.isnull().sum().sum())

User_ID                         0
Age                            23
Gender                         10
Country                        10
User_Type                       0
Education_Level                13
Profession                     15
Monthly_Income                 10
AI_Tool                        11
AI_Purpose                     13
AI_Usage_Hours_Per_Day          7
Tasks_Performed                18
Monthly_AI_Cost                12
Productivity_Score             16
Accuracy_Rating                15
Satisfaction_Score              7
Time_Saved_Hours_Per_Week      10
Work_or_Study_Hours_Per_Day     9
Would_Recommend                 5
dtype: int64
Total missing: 204


# Convert Empty Values in NAN

In [3]:
df = df.replace(r"^\s*$", np.nan, regex=True)

# Now we again check empty values 

In [4]:
print(df.isnull().sum())

User_ID                         0
Age                            23
Gender                         10
Country                        10
User_Type                       0
Education_Level                13
Profession                     15
Monthly_Income                 10
AI_Tool                        11
AI_Purpose                     13
AI_Usage_Hours_Per_Day          7
Tasks_Performed                18
Monthly_AI_Cost                12
Productivity_Score             16
Accuracy_Rating                15
Satisfaction_Score              7
Time_Saved_Hours_Per_Week      10
Work_or_Study_Hours_Per_Day     9
Would_Recommend                 5
dtype: int64


In [5]:
numeric_cols = [
    "Age",
    "Monthly_Income",
    "AI_Usage_Hours_Per_Day",
    "Monthly_AI_Cost",
    "Productivity_Score",
    "Accuracy_Rating",
    "Satisfaction_Score",
    "Time_Saved_Hours_Per_Week",
    "Work_or_Study_Hours_Per_Day"
]

for col in numeric_cols:
    s = (
        df[col].astype("string")
        .str.strip()
        .str.replace(
            r"(?i)\s*(hrs?|hours?)\s*$", "",
            regex=True
        )
        .str.replace(r"[$,]", "", regex=True)
    )

    df[col] = pd.to_numeric(s, errors="coerce")

In [6]:
df["Age"].fillna(df["Age"].mean())

TypeError: Invalid value '30.182509505703422' for dtype 'Int64'

In [7]:
df["Age"] = pd.to_numeric(
    df["Age"], errors="coerce"
).astype("float64")

df["Age"] = df["Age"].fillna(
    df["Age"].mean()
)

print(df["Age"].isnull().sum())
print(df["Age"].dtype)

0
float64


In [8]:
numeric_cols = [
    "Age",
    "Monthly_Income",
    "AI_Usage_Hours_Per_Day",
    "Monthly_AI_Cost",
    "Productivity_Score",
    "Accuracy_Rating",
    "Satisfaction_Score",
    "Time_Saved_Hours_Per_Week",
    "Work_or_Study_Hours_Per_Day"
]

for col in numeric_cols:
    if col in df.columns:

        # Convert columns to decimal-compatible numeric type
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        ).astype("float64")

        # Fill missing values using median
        if df[col].notna().any():
            df[col] = df[col].fillna(
                df[col].median()
            )

print("Missing values after filling:")
print(df[numeric_cols].isnull().sum())

Missing values after filling:
Age                            0
Monthly_Income                 0
AI_Usage_Hours_Per_Day         0
Monthly_AI_Cost                0
Productivity_Score             0
Accuracy_Rating                0
Satisfaction_Score             0
Time_Saved_Hours_Per_Week      0
Work_or_Study_Hours_Per_Day    0
dtype: int64


In [9]:
for col in numeric_cols:
    print(col, ":", (df[col] == 0).sum())

Age : 0
Monthly_Income : 34
AI_Usage_Hours_Per_Day : 0
Monthly_AI_Cost : 166
Productivity_Score : 0
Accuracy_Rating : 0
Satisfaction_Score : 0
Time_Saved_Hours_Per_Week : 8
Work_or_Study_Hours_Per_Day : 0


In [10]:
# Example only: verified zero-as-missing columns
zero_missing_cols = []

for col in zero_missing_cols:
    df.loc[df[col] == 0, col] = np.nan
    df[col] = df[col].fillna(df[col].median())

In [11]:
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)

Duplicates: 12
Shape after cleaning: (288, 19)


In [12]:
print("Missing values after cleaning:")
print(df.isnull().sum())

print(
    "Total missing values:",
    df.isnull().sum().sum()
)

print("Final dataset shape:", df.shape)

print(df.info())

Missing values after cleaning:
User_ID                         0
Age                             0
Gender                         10
Country                        10
User_Type                       0
Education_Level                12
Profession                     14
Monthly_Income                  0
AI_Tool                        10
AI_Purpose                     13
AI_Usage_Hours_Per_Day          0
Tasks_Performed                17
Monthly_AI_Cost                 0
Productivity_Score              0
Accuracy_Rating                 0
Satisfaction_Score              0
Time_Saved_Hours_Per_Week       0
Work_or_Study_Hours_Per_Day     0
Would_Recommend                 5
dtype: int64
Total missing values: 91
Final dataset shape: (288, 19)
<class 'pandas.DataFrame'>
Index: 288 entries, 0 to 299
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   User_ID                      288 non-null   

In [13]:
categorical_cols = [
    "Gender",
    "Country",
    "Education_Level",
    "Profession",
    "AI_Tool",
    "AI_Purpose",
    "Would_Recommend"
]

for col in categorical_cols:
    if df[col].isnull().sum() > 0:

        mode_value = df[col].mode(dropna=True)

        if not mode_value.empty:
            df[col] = df[col].fillna(mode_value.iloc[0])
        else:
            df[col] = df[col].fillna("Unknown")

In [14]:
df["Tasks_Performed"] = df["Tasks_Performed"].fillna(
    "Not specified"
)

In [15]:
print("Missing values after cleaning:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nFinal dataset shape:")
print(df.shape)

Missing values after cleaning:
User_ID                        0
Age                            0
Gender                         0
Country                        0
User_Type                      0
Education_Level                0
Profession                     0
Monthly_Income                 0
AI_Tool                        0
AI_Purpose                     0
AI_Usage_Hours_Per_Day         0
Tasks_Performed                0
Monthly_AI_Cost                0
Productivity_Score             0
Accuracy_Rating                0
Satisfaction_Score             0
Time_Saved_Hours_Per_Week      0
Work_or_Study_Hours_Per_Day    0
Would_Recommend                0
dtype: int64

Total missing values:
0

Final dataset shape:
(288, 19)


In [16]:
df.to_csv(
    "AI_Usage_and_Impact_Cleaned.csv",
    index=False
)

print("Dataset cleaned and saved successfully!")

Dataset cleaned and saved successfully!


In [17]:
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())
print(df.head())

Duplicate rows: 0
Missing values: 0
  User_ID   Age  Gender         Country     User_Type Education_Level  \
0   U0269  30.0    Male  United Kingdom  Professional        Master's   
1   U0217  30.0    Male   United States  Professional      Bachelor's   
2   U0152  17.0    Male          Canada       Student      Bachelor's   
3   U0264  35.0  FEMALE          Brazil  Professional      Bachelor's   
4   U0263  19.0  Female          Canada       Student   Undergraduate   

                                        Profession  Monthly_Income  \
0                                 Business Analyst          5421.0   
1                      Customer Support Specialist          8903.0   
2                         Computer Science Student           402.0   
3  Desenvolvedora de Software / Software Developer          1367.0   
4                                  Biology Student           438.0   

         AI_Tool        AI_Purpose  AI_Usage_Hours_Per_Day  \
0        ChatGPT       Office Work        

In [18]:
df = df.reset_index(drop=True)